# Fine-Tuning DistilBERT for Text Classification

In this notebook, we will explore **what fine-tuning is**, **why we chose DistilBERT**, and how to fully train a model for text classification using the Hugging Face ecosystem.

## 1. What is Fine-Tuning?
Instead of training a model from scratch, fine-tuning takes a model that has already been pre-trained on a massive dataset (like all of Wikipedia) and gently adjusts its internal weights to perform a specific new task (like sentiment analysis). It saves massive amounts of time and compute cost!

## 2. Why DistilBERT?
DistilBERT is a smaller, faster, and lighter version of BERT. Using a technique called 'knowledge distillation', it retains 97% of BERT's performance while being 40% smaller and 60% faster. This makes it perfect for experimenting on personal computers or achieving fast inference in production.

In [ ]:
# Run this cell to install dependencies (if not already installed).
!pip install transformers datasets evaluate accelerate torch

## 3. Dataset Preparation
We will use the `datasets` library to load a sample dataset (IMDB reviews for sentiment classification) and tokenize the text.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# 1. Load the dataset (We'll use a small subset for speed)
dataset = load_dataset('imdb')

# 2. Load the DistilBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

# 3. Tokenize the input text
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Dataset tokenized successfully!")

## 4. Model Training
Now we use the `Trainer` class from Hugging Face which handles the training loop for us. We load the model `DistilBertForSequenceClassification` with 2 output labels (positive, negative).

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# Load metrics
metric = evaluate.load('accuracy')
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Load Model
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

# Define Training Arguments
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'].shuffle(seed=42).select(range(1000)), # Subset for speed
    eval_dataset=tokenized_datasets['test'].shuffle(seed=42).select(range(1000)),
    compute_metrics=compute_metrics,
)

# Run Training
# trainer.train()  # Uncomment to start the actual fine-tuning process!